In [48]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from io import StringIO

In [45]:
prem_url = "https://fbref.com/en/comps/9/Premier-League-Stats"


prem_data = requests.get(prem_url)

soup = BeautifulSoup(prem_data.text, 'html.parser')

# Grabs all the team links from a premier league season 
def get_group_link(soup):
    # Finds the first table which contains the links of the teams
    table = soup.find('table')
    # Gets the links from the table 
    a_tags_with_href = table.find_all('a', href=True)
    # Regular expression to get the correct links
    pattern = re.compile(r'^/en/squads/.+/.+$') 
    filtered_links = set()  # Use a set to ensure uniqueness
    baseUrl = "https://fbref.com"
    # Checks the href against the regex, if it matches append (baseUrl + href) to filtered_links
    
    for a in a_tags_with_href:
        href = a['href']
        if pattern.match(href):
            filtered_links.add(baseUrl + href)
    # Change it into a list so it can be indexed later 
    filtered_links = list(filtered_links)
    return filtered_links

team_urls = get_group_link(soup)
team_urls


['https://fbref.com/en/squads/d07537b9/2021-2022/Brighton-and-Hove-Albion-Stats',
 'https://fbref.com/en/squads/943e8050/2021-2022/Burnley-Stats',
 'https://fbref.com/en/squads/33c895d4/2021-2022/Southampton-Stats',
 'https://fbref.com/en/squads/cd051869/2021-2022/Brentford-Stats',
 'https://fbref.com/en/squads/b8fd03ef/2021-2022/Manchester-City-Stats',
 'https://fbref.com/en/squads/7c21e445/2021-2022/West-Ham-United-Stats',
 'https://fbref.com/en/squads/1c781004/2021-2022/Norwich-City-Stats',
 'https://fbref.com/en/squads/361ca564/2021-2022/Tottenham-Hotspur-Stats',
 'https://fbref.com/en/squads/cff3d9bb/2021-2022/Chelsea-Stats',
 'https://fbref.com/en/squads/5bfb9659/2021-2022/Leeds-United-Stats',
 'https://fbref.com/en/squads/822bd0ba/2021-2022/Liverpool-Stats',
 'https://fbref.com/en/squads/47c64c55/2021-2022/Crystal-Palace-Stats',
 'https://fbref.com/en/squads/2abfe087/2021-2022/Watford-Stats',
 'https://fbref.com/en/squads/8cec06e1/2021-2022/Wolverhampton-Wanderers-Stats',
 'http

In [50]:
years = list(range(2024, 2022, -1))

all_matches = []
standings_url = "https://fbref.com/en/comps/9/Premier-League-Stats"

import time
for year in years:
    data = requests.get(standings_url)
    soup = BeautifulSoup(data.text, 'html.parser')
    team_urls = get_group_link(soup)
    
    # Adjusts the standing_urls every loop, so that it gets the previous season 
    previous_season = soup.select("a.prev")[0].get("href")
    standings_url = f"https://fbref.com{previous_season}"
    
    for team_url in team_urls:
        # Get team name
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        # Gets the html data for a team
        data = requests.get(team_url)
        matches = pd.read_html(StringIO(data.text), match="Scores & Fixtures")[0]
        soup = BeautifulSoup(data.text)
        links = [l.get("href") for l in soup.find_all('a')]
        links = [l for l in links if l and 'all_comps/shooting/' in l]
        data = requests.get(f"https://fbref.com{links[0]}")
        shooting = pd.read_html(StringIO(data.text), match="Shooting")[0]
        shooting.columns = shooting.columns.droplevel()
        try:
            team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]], on="Date")
        except ValueError:
            continue
        team_data = team_data[team_data["Comp"] == "Premier League"]
        
        team_data["Season"] = year
        team_data["Team"] = team_name
        all_matches.append(team_data)
        # Might get rate limited, so adjust delay as necessary 
        time.sleep(30)
all_matches

[          Date   Time            Comp         Round  Day Venue Result  GF  GA  \
 0   2023-08-12  15:00  Premier League   Matchweek 1  Sat  Home      W   4   1   
 1   2023-08-19  15:00  Premier League   Matchweek 2  Sat  Away      W   4   1   
 2   2023-08-26  17:30  Premier League   Matchweek 3  Sat  Home      L   1   3   
 3   2023-09-02  17:30  Premier League   Matchweek 4  Sat  Home      W   3   1   
 4   2023-09-16  15:00  Premier League   Matchweek 5  Sat  Away      W   3   1   
 6   2023-09-24  14:00  Premier League   Matchweek 6  Sun  Home      W   3   1   
 8   2023-09-30  12:30  Premier League   Matchweek 7  Sat  Away      L   1   6   
 10  2023-10-08  14:00  Premier League   Matchweek 8  Sun  Home      D   2   2   
 11  2023-10-21  15:00  Premier League   Matchweek 9  Sat  Away      L   1   2   
 13  2023-10-29  14:00  Premier League  Matchweek 10  Sun  Home      D   1   1   
 14  2023-11-04  15:00  Premier League  Matchweek 11  Sat  Away      D   1   1   
 16  2023-11-12 